# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print general information
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll list the available record sets (`RecordSet`), their fields, and available columns/fields, focusing on `@id` references for clarity and citable reproducibility.

In [ ]:
from pprint import pprint

# List all record sets in the dataset
print("Record sets available in this dataset:")
record_sets = list(dataset.record_sets)
record_set_ids = [r['@id'] for r in record_sets]
for record_set in record_sets:
    rid = record_set['@id']
    name = record_set.get('name', rid)
    print(f"- {name} (@id: {rid})")

# For each record set, print out their fields (columns)
print("\nFields and columns per record set:")
record_set_fields = {}
for record_set in record_sets:
    rid = record_set['@id']
    print(f"\nRecord set: {rid}")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]  # Ensure always a list
    record_set_fields[rid] = []
    for field in fields:
        fid = field['@id'] if isinstance(field, dict) else field
        record_set_fields[rid].append(fid)
        print(f"  - Field @id: {fid}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For demonstration, we'll extract data from the first available record set.

In [ ]:
# Choose a record set for extraction (here, use the first)
selected_record_set_id = record_set_ids[0] if record_set_ids else None

if selected_record_set_id:
    # Load all records for this record set
    records = list(dataset.records(record_set=selected_record_set_id))
    df = pd.DataFrame(records)
    print(f"Loaded DataFrame for record set {selected_record_set_id} with columns:")
    print(df.columns.tolist())
    display_cols = df.columns.tolist()[:10]
    df[display_cols].head()
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will:
- Select a numeric field to filter and normalize (e.g., molecular weight, MW).
- Remove records with missing or invalid values.
- Group records by a categorical field, if available (e.g., peptide type or protein description).

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Try to find a numeric field (commonly 'MW' for molecular weight, or 'Coverage(%)')
numeric_fields = [col for col in df.columns if df[col].dtype in [int, float, np.int64, np.float64]]
if not numeric_fields:
    # Try for objects that look like numbers
    possible_numeric = [col for col in df.columns if df[col].apply(lambda v: str(v).replace('.','',1).isdigit() if pd.notnull(v) else False).sum() > 0]
    # Take the first possible numeric field
    numeric_field = possible_numeric[0] if possible_numeric else df.columns[0]
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
else:
    numeric_field = numeric_fields[0]

print(f"Using numeric field for analysis: {numeric_field}")

# Filter out invalid values
threshold = df[numeric_field].mean() if np.isfinite(df[numeric_field].mean()) else 0
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records where {numeric_field} > {threshold:.2f} (mean): {filtered_df.shape[0]} rows")
print(filtered_df[[numeric_field]].head())

# Normalize the field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to pick a group field (string/categorical)
group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
group_field = group_fields[0] if group_fields else None
if group_field:
    print(f"\nGrouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
plt.title(f'Histogram of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If group_field is available, plot top 10 group means
if group_field:
    plt.figure(figsize=(10,6))
    top10 = grouped_df.head(10)
    sns.barplot(x=numeric_field, y=group_field, data=top10, palette="viridis")
    plt.title(f'Top 10 {group_field} by mean {numeric_field}')
    plt.xlabel(f'Mean {numeric_field}')
    plt.ylabel(group_field)
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR<sup>2</sup> dataset _Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells_ using the `mlcroissant` library. We reviewed record sets and their field `@id`s, loaded data, applied processing steps (filtering, normalization, grouping), and visualized protein abundance or molecular weight distributions. This workflow can be adapted for in-depth domain-specific analysis and integration into larger FAIR data pipelines.